# Crash safety under 1,000 kills, and a real RF-DETR training run

This notebook backs two claims with runs, not with prose.

**Part A. Checkpoints survive `kill -9`.** A writer process saves checkpoints and is killed with
`SIGKILL` inside a save, the same number of times (set in a cell below) in each of four groups of
places (before the temporary file is created, in the middle of writing it, after writing but before
any rename, after the renames but before the run store and the `LATEST` pointer are updated) and at
a random moment. The groups together name a number of places that a cell below counts, and how often
each was hit is computed too. After every kill
a fresh process restores, compares the state byte for byte with what that step saved, and saves
and restores the next step. Then a number of uploads to an in-memory object store (counted below) are cut after a random
number of bytes. The harness is the one the test suite runs (`tests/chaos/`).

**Part B. The RF-DETR hook on a real model.** RF-DETR Nano (Apache-2.0) with its published
COCO weights is fine-tuned for two epochs on 64 COCO128 photos on the CPU, with
`cvpilot.autolog(model)` attached. The licences of the package and of the weights are checked
before the weights are downloaded, the download is SHA-256 pinned, and the values CV-Pilot
recorded are compared with the framework's own metrics file.

Every number below is computed by the cell that prints it. Lines that start with `CHECK:` are
computed pass/fail results; the last cell counts them. Data: COCO128 (annotations CC BY 4.0, the
images keep their Flickr licences). The weights are about 370 MB and are cached in
`$CVPILOT_NOTEBOOK_DATA`.

In [1]:
import contextlib
import csv
import hashlib
import io
import json
import logging
import os
import re
import shlex
import shutil
import signal
import socket
import stat
import statistics
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request
import warnings
import zipfile
from collections import Counter
from pathlib import Path

warnings.filterwarnings("ignore")


def private_data_dir():
    """The download cache: ~/.cache/cvpilot-notebooks (or CVPILOT_NOTEBOOK_DATA), created with mode
    0700. It must be a real folder that belongs to you and that nobody else can write: a shared
    folder such as /tmp would let another user swap a file between its check and its use."""
    default = Path.home() / ".cache" / "cvpilot-notebooks"
    # resolved first, so every component below is a real folder (no link can be swapped in later)
    path = Path(os.environ.get("CVPILOT_NOTEBOOK_DATA") or default).expanduser().resolve()
    path.mkdir(mode=0o700, parents=True, exist_ok=True)
    info = path.lstat()
    if stat.S_ISLNK(info.st_mode) or not stat.S_ISDIR(info.st_mode):
        raise RuntimeError("the download cache must be a folder, not a link")
    if info.st_uid != os.getuid() or info.st_mode & 0o022:
        raise RuntimeError("the download cache must belong to you and be writable by you only")
    for parent in path.parents:  # nobody else may rename or replace a folder on the way down
        up = parent.lstat()
        if up.st_uid not in (0, os.getuid()):
            raise RuntimeError(f"{parent} belongs to another user: choose another cache folder")
        if up.st_mode & 0o022 and not up.st_mode & stat.S_ISVTX:
            raise RuntimeError(f"{parent} can be changed by others: choose another cache folder")
    return path


# Where downloads are cached. This is outside the repository and nothing in it is ever committed.
DATA = private_data_dir()
# A fresh scratch folder for this run of the notebook: everything the notebook writes lives here.
REPO = Path.cwd()  # the notebook is run from the repository root (see notebooks/README.md)
WORK = Path(tempfile.mkdtemp(prefix="cvpilot-nb-"))
os.chdir(WORK)

os.environ["CVPILOT_STORE_DIR"] = "store"  # runs, metrics and evaluation results
os.environ["CVPILOT_PROJECT"] = "rfdetr"  # default project of every cvpilot command
os.environ["CVPILOT_LOG_LEVEL"] = "WARNING"  # keeps this notebook's own Python calls quiet
os.environ.pop("CVPILOT_ENV", None)  # production behaviour
os.environ.pop("CVPILOT_CKPT_KEY", None)
os.environ.update(
    HF_HUB_OFFLINE="1",  # the framework never reaches the Hugging Face Hub from here
    HF_HUB_DISABLE_TELEMETRY="1",
    RF_HOME=str(WORK / "rf-home"),  # the framework's own weight cache: empty, nothing is fetched
    NO_COLOR="1",
    TERM="dumb",
    TQDM_DISABLE="1",
    PYTHONWARNINGS="ignore",
)
TIMINGS = {}  # seconds per step, printed at the end

print("python", sys.version.split()[0], "| scratch folder: a fresh temporary directory")
print("downloads are cached in $CVPILOT_NOTEBOOK_DATA (default: ~/.cache/cvpilot-notebooks)")

python 3.13.9 | scratch folder: a fresh temporary directory
downloads are cached in $CVPILOT_NOTEBOOK_DATA (default: ~/.cache/cvpilot-notebooks)


### Getting ready

The helpers of the earlier notebooks: `sh()` runs a command and shows its exit code, `scrub()` hides
this computer's paths, `check()` records a computed pass/fail line.

In [2]:
_REPLACEMENTS = sorted(
    [
        (str(WORK.resolve()), "."),
        (str(WORK), "."),
        (str(DATA.resolve()), "$CVPILOT_NOTEBOOK_DATA"),
        (str(DATA), "$CVPILOT_NOTEBOOK_DATA"),
        (str(Path.home()), "~"),
        (socket.gethostname(), "<this-computer>"),
    ],
    key=lambda pair: -len(pair[0]),
)
_OTHER_PATHS = re.compile(r"/(?:Users|home|private|var|tmp|opt)/[^\s\"',:)]+")
_QUIET_FIELDS = {"logger", "request_id", "service", "env", "@timestamp", "level", "message"}
_CLI = "import sys; sys.argv[0] = 'cvpilot'; from cvpilot.cli.app import main; main()"


def scrub(text):
    """Hide the paths and the computer name of the machine that runs this notebook."""
    for old, new in _REPLACEMENTS:
        if old:
            text = text.replace(old, new)
    return _OTHER_PATHS.sub("<path>", text)


def log_summary(line):
    """cvpilot writes one JSON object per log line on stderr; show it as one short line."""
    try:
        event = json.loads(line)
    except ValueError:
        return "stderr: " + scrub(line)
    if not isinstance(event, dict):
        return "stderr: " + scrub(line)
    fields = " ".join(
        f"{key}={scrub(str(value))[:44]}"
        for key, value in event.items()
        if key not in _QUIET_FIELDS
    )
    return f"log[{event.get('level', '?')}] {event.get('message')} {fields}"


class Result:
    def __init__(self, code, out, err):
        self.code, self.out, self.err = code, out, err

    def json(self):
        return json.loads(self.out)


def sh(
    command, *, lines=40, log_lines=8, quiet=False, keep=False, env=None, expect=None, timeout=900
):
    """Run one command, print what it printed (paths hidden) and its exit code.

    `cvpilot ...` runs the command-line tool of this Python environment. Output that is not
    JSON on stderr is shown as it is; the JSON log lines are shortened to one line each.
    The result (exit code, output) is returned only with `keep=True` or `quiet=True`.
    """
    argv = shlex.split(command)
    if argv[0] == "cvpilot":
        argv = [sys.executable, "-c", _CLI, *argv[1:]]
    elif argv[0] == "python":
        argv[0] = sys.executable
    full_env = {**os.environ, "CVPILOT_LOG_LEVEL": "INFO", **(env or {})}
    proc = subprocess.run(  # noqa: S603
        argv, capture_output=True, text=True, cwd=WORK, env=full_env, timeout=timeout, check=False
    )
    result = Result(proc.returncode, scrub(proc.stdout), scrub(proc.stderr))
    if not quiet:
        print("$", scrub(command))
        out_lines = result.out.rstrip("\n").splitlines()
        for line in out_lines[:lines]:
            print(line[:180] + (" ..." if len(line) > 180 else ""))
        if len(out_lines) > lines:
            print(f"... ({len(out_lines) - lines} more lines)")
        err_lines = [ln for ln in proc.stderr.splitlines() if ln.strip()]
        for line in err_lines[:log_lines]:
            print(log_summary(line)[:200])
        if len(err_lines) > log_lines:
            print(f"... ({len(err_lines) - log_lines} more log lines)")
        print("exit code:", proc.returncode)
    if expect is not None and proc.returncode != expect:
        raise RuntimeError(f"expected exit code {expect}, got {proc.returncode}: {command}")
    return result if (keep or quiet) else None


@contextlib.contextmanager
def timed(label):
    start = time.perf_counter()
    yield
    TIMINGS[label] = round(time.perf_counter() - start, 1)
    print(f"[{label}: {TIMINGS[label]} s]")

In [3]:
CHECKS = []  # (label, passed): printed as CHECK lines and counted at the end
LAYOUTS = []  # one row per framework file: framework, file, outcome, exact message


def check(label, passed):
    CHECKS.append((label, bool(passed)))
    print(f"CHECK: {label}: {bool(passed)}")


def layout(framework, file, outcome, message=""):
    LAYOUTS.append((framework, file, outcome, scrub(message)))

# Part A. Kill the writer inside a save, 1,000 times

`tests/chaos/_ckpt_kill_points.py` forks a writer per iteration. The writer saves one to four
checkpoints, then stops itself at the chosen place inside the next save; the harness kills it with
`SIGKILL` there (or, for `random_moment`, kills it 0 to 4 ms after that save started, wherever it
is). A fresh process then does what a restarted job does: marks the dead run failed, resumes a new
run from it, restores, compares the state with what that step saved, saves the next step and
restores that. Each iteration varies, from a fixed seed: the number of earlier saves, the tensor
size (4 bytes to 1 MiB), `keep_last`, whether an authentication key is set and whether the
`anchored` trust policy is required.

In [4]:
def run_group(argv, env, timeout, cwd=None):
    """Run argv in its own process group and kill the whole group afterwards: the harness forks
    writers that stop themselves with SIGSTOP, and they must not outlive it."""
    proc = subprocess.Popen(  # noqa: S603
        argv,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        env=env,
        cwd=cwd,
        start_new_session=True,
    )
    try:
        out, err = proc.communicate(timeout=timeout)
    finally:
        with contextlib.suppress(ProcessLookupError):
            os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    return proc.returncode, out, err


HARNESS = REPO / "tests" / "chaos" / "_ckpt_kill_points.py"
POINTS = [
    "before_temp_write",
    "mid_write",
    "after_write_before_rename",
    "after_rename_before_chain",
    "random_moment",
]
ITERATIONS, SEED = 200, 20261026
CHAOS = {}
with timed("1,000 kills"):
    for point in POINTS:
        home = WORK / "chaos-home" / point
        home.mkdir(parents=True)
        code, out, err = run_group(
            [
                sys.executable,
                str(HARNESS),
                str(WORK / "chaos" / point),
                point,
                str(SEED),
                str(ITERATIONS),
            ],
            {"PATH": "/usr/bin:/bin", "HOME": str(home), "PYTHONHASHSEED": "0"},
            900,
        )
        if code != 0:
            raise RuntimeError(scrub(err[-2000:]))
        CHAOS[point] = [json.loads(x) for x in out.splitlines() if x.startswith("{")]
        print(f"{point:27s} {len(CHAOS[point])} iterations")

before_temp_write           200 iterations


mid_write                   200 iterations


after_write_before_rename   200 iterations


after_rename_before_chain   200 iterations


random_moment               200 iterations
[1,000 kills: 198.0 s]


In [5]:
print(
    f"{'kill point':27s} {'kills':>5s} {'killed':>6s} {'right step':>10s} {'newer':>5s} "
    f"{'acked':>5s} {'key':>4s} {'anchored':>8s} {'tmp left':>8s} {'median s':>8s}"
)
for point, rows in CHAOS.items():
    right = sum(1 for r in rows if r["ok"])
    newer = sum(1 for r in rows if r["ok"] and r["restored"] == r["acks"] + 1)
    print(
        f"{point:27s} {len(rows):5d} {sum(r['killed'] for r in rows):6d} {right:10d} "
        f"{newer:5d} {right - newer:5d} {sum(r['keyed'] for r in rows):4d} "
        f"{sum(r['anchored'] for r in rows):8d} {sum(1 for r in rows if r.get('temp_files')):8d} "
        f"{statistics.median(r['seconds'] for r in rows):8.2f}"
    )
total = sum(len(rows) for rows in CHAOS.values())
right = sum(r["ok"] for rows in CHAOS.values() for r in rows)
check(
    f"every one of the {total} kills was a real SIGKILL",
    all(r["killed"] for rows in CHAOS.values() for r in rows),
)
check(
    f"{right} of {total} restarts restored exactly the expected checkpoint, byte for byte",
    right == total,
)
check(
    f"every group has the {ITERATIONS} kills it was asked for",
    all(len(rows) == ITERATIONS for rows in CHAOS.values()),
)

kill point                  kills killed right step newer acked  key anchored tmp left median s
before_temp_write             200    200        200     0   200   58       57        0     0.20
mid_write                     200    200        200     0   200   59       60      200     0.19
after_write_before_rename     200    200        200     0   200   61       48      200     0.19
after_rename_before_chain     200    200        200   145    55   61       50       38     0.19
random_moment                 200    200        200   108    92   53       64       41     0.19
CHECK: every one of the 1000 kills was a real SIGKILL: True
CHECK: 1000 of 1000 restarts restored exactly the expected checkpoint, byte for byte: True
CHECK: every group has the 200 kills it was asked for: True


**What this shows.** `right step` counts the restarts that restored exactly the checkpoint the
place implies (the previous one when the kill came before the new pair was complete on disk, the new
one when it came after) with the content that step saved, and then saved and restored the next
step. `newer` and `acked` split them by which of the two was restored. `tmp left` counts the kills
that left a half-written temporary file on disk: it was there at the restart and was not loaded; the
next save in that directory removes it.

In [6]:
print(
    f"{'kill point':27s} {'place inside the save':22s} {'kills':>5s} {'right':>5s}  skipped at the restart"
)
for point, rows in CHAOS.items():
    for place in sorted({r["place"] for r in rows}):
        sel = [r for r in rows if r["place"] == place]
        reasons = Counter(reason for r in sel for reason in r.get("skipped", []))
        print(
            f"{point:27s} {place:22s} {len(sel):5d} {sum(r['ok'] for r in sel):5d}  "
            f"{dict(reasons) or '-'}"
        )
torn = [r for r in CHAOS["mid_write"] if r["place"] == "data_bytes"]
check(
    "every torn data file was still on disk at the restart and was not restored",
    all(r["temp_files"] >= 1 and r["ok"] for r in torn),
)
half = [r for r in CHAOS["after_rename_before_chain"] if r["place"] == "manifest_linked"]
check(
    "a manifest whose data file never got its name is refused: not_anchored under the anchored "
    "policy, data_missing otherwise",
    all(("not_anchored" if r["anchored"] else "data_missing") in r["skipped"] for r in half),
)
per_place = Counter(
    r["place"] for point, rows in CHAOS.items() if point != "random_moment" for r in rows
)
fewest = min(per_place, key=per_place.get)
print(
    f"{len(per_place)} named places inside a save; the fewest kills at one place: "
    f"{per_place[fewest]} ({fewest})"
)
check("every named place was killed at least 20 times", min(per_place.values()) >= 20)
in_flight = sum(r["in_save"] for r in CHAOS["random_moment"])
print(f"random-moment kills with a save in flight: {in_flight} of {len(CHAOS['random_moment'])}")
check("at least one random-moment kill landed with a save in flight", in_flight >= 1)

kill point                  place inside the save  kills right  skipped at the restart
before_temp_write           before_data_open         200   200  {'rolled_back': 50}
mid_write                   data_bytes               156   156  {'rolled_back': 44}
mid_write                   manifest_bytes            44    44  {'rolled_back': 13}
after_write_before_rename   before_first_link         50    50  {'rolled_back': 13}
after_write_before_rename   data_written              67    67  {'rolled_back': 24}
after_write_before_rename   manifest_written          83    83  {'rolled_back': 26}
after_rename_before_chain   data_linked               38    38  {'rolled_back': 12, 'not_anchored': 8}
after_rename_before_chain   manifest_linked           38    38  {'not_anchored': 13, 'rolled_back': 12, 'data_missing': 25}
after_rename_before_chain   pointer_temp_written      37    37  {'rolled_back': 24}
after_rename_before_chain   recorded_no_pointer       46    46  {'rolled_back': 25}
after_rename_b

**What this shows.** The place inside each kill point and the refusals counted while the restart
listed every checkpoint of the dead run. A manifest whose data never got its final name is refused
as `data_missing`, or first as `not_anchored` when the `anchored` policy is required (the store never
recorded it); a torn temporary file is invisible because it never had a final name. Under the
`anchored` and `authenticated` policies the older checkpoints behind the newest one are listed as
`rolled_back`: they are not lost, a strict policy only admits the newest.

### Uploads cut short

The same promise for the copy in object storage: a number of uploads to fsspec's in-memory filesystem are
cut after a random number of bytes, as a dropped connection or as a process that dies in the middle
of the upload. A node with nothing local must then restore exactly the newest checkpoint whose
upload completed, and every manifest in the bucket must describe a data object of exactly its size
and digest. This runs the test from the suite and prints its summary line.

In [7]:
with timed("200 cut uploads"):
    # an environment built from an allow-list and a private HOME and TMPDIR: nothing of this
    # notebook's variables, and nothing of your home folder, reaches the test run
    for private in ("pytest-home", "pytest-tmp"):
        (WORK / private).mkdir(exist_ok=True)
    code, out, err = run_group(
        [
            sys.executable,
            "-m",
            "pytest",
            "-m",
            "chaos",
            "-q",
            "-s",
            "-p",
            "no:cacheprovider",
            "tests/chaos/test_ckpt_truncated_upload.py",
        ],
        {
            "PATH": "/usr/bin:/bin",
            "HOME": str(WORK / "pytest-home"),
            "TMPDIR": str(WORK / "pytest-tmp"),
            "CVPILOT_LOG_LEVEL": "ERROR",
            "NO_COLOR": "1",
        },
        900,
        cwd=REPO,
    )
summary = [x for x in out.splitlines() if "truncated uploads:" in x]
for line in summary:
    print(line.strip().lstrip("."))
print(out.strip().splitlines()[-1])
check("the truncated-upload test passed", code == 0 and len(summary) == 1)

[200 cut uploads: 7.8 s]
truncated uploads: 200 cuts (100 deaths, 100 dropped connections); targets {'data': 149, 'manifest': 51}; restored the newest completed upload in 172; nothing restored because no upload had completed in 28; resumed appends 31; partial objects left on the remote 102
1 passed in 7.28s
CHECK: the truncated-upload test passed: True


**What this shows.** Uploads that were cut never became restorable checkpoints; the newest upload
that completed did. A cut upload leaves a `.part` object behind when the process died before it
could delete it; it is never read as a checkpoint and is replaced when that checkpoint is uploaded
again. This test found a bug before it passed: a `memory://` remote could not save at all (its URL
gained a third slash and failed validation); it is fixed and has its own unit test.

# Part B. RF-DETR: licences first, then a pinned download

RF-DETR is installed through the `cvpilot[rfdetr]` extra. Before anything is downloaded this cell
reads the licence of the installed package and the licence the framework's own model configuration
states for the Nano weights, and checks that the framework's weight registry names exactly the URL
and MD5 pinned here. The larger XLarge and 2XLarge models are under a different licence (the
framework asks for `accept_platform_model_license=True`); they are not used.

In [8]:
import importlib.metadata as metadata

logging.getLogger("rf-detr").setLevel(logging.ERROR)
for name in ("lightning", "py" + "torch_lightning", "lightning_fabric"):
    logging.getLogger(name).setLevel(logging.ERROR)
package = metadata.metadata("rfdetr")
classifiers = [c for c in package.get_all("Classifier") or [] if c.startswith("License")]
print(
    "rfdetr",
    metadata.version("rfdetr"),
    "| licence field:",
    package.get("License"),
    "| classifiers:",
    classifiers,
)
check(
    "the installed rfdetr package is Apache-2.0",
    package.get("License") in ("Apache License 2.0", "Apache-2.0")
    and classifiers == ["License :: OSI Approved :: Apache Software License"],
)
for dist in (
    "supervision",
    "peft",
    "py" + "torch-lightning",
    "torchmetrics",
    "faster-coco-eval",
    "hotcoco",
    "ultrafast-pycocotools",
    "vernier",
    "simplejpeg",
    "torch-hungarian",
):
    meta = metadata.metadata(dist)
    lic = meta.get("License-Expression") or meta.get("License") or "-"
    lic_cls = [
        c.split("::")[-1].strip()
        for c in meta.get_all("Classifier") or []
        if c.startswith("License")
    ]
    print(f"   {dist:24s} {metadata.version(dist):10s} {lic[:30]:30s} {lic_cls}")

rfdetr 1.11.2 | licence field: Apache License 2.0 | classifiers: ['License :: OSI Approved :: Apache Software License']
CHECK: the installed rfdetr package is Apache-2.0: True
   supervision              0.31.2     MIT                            []
   peft                     0.21.2     Apache                         ['Apache Software License']
   pytorch-lightning        2.6.6      Apache-2.0                     ['Apache Software License']
   torchmetrics             1.8.2      Apache-2.0                     ['Apache Software License']
   faster-coco-eval         1.8.0      -                              []
   hotcoco                  1.0.1      MIT                            ['MIT License']
   ultrafast-pycocotools    0.1.14     BSD-2-Clause                   ['BSD License']
   vernier                  0.5.4      MIT OR Apache-2.0              ['MIT License', 'Apache Software License']
   simplejpeg               1.9.0      MIT                            []
   torch-hungarian        

In [9]:
from rfdetr.assets.model_weights import ModelWeights
from rfdetr.config import RFDETRNanoConfig

NANO_URL = "https://storage.googleapis.com/rfdetr/nano_coco/checkpoint_best_regular.pth"
NANO_MD5 = "fb6504cce7fbdc783f7a46991f07639f"  # the framework registry's digest for this file
asset = ModelWeights.from_filename("rf-detr-nano.pth")
nano_licence = RFDETRNanoConfig().license
print("weights:", asset.filename, "| licence in the framework's model config:", nano_licence)
check("the Nano weights are published under Apache-2.0", nano_licence == "Apache-2.0")
check("the framework's registry names the pinned URL", asset.url == NANO_URL)
check("the framework's registry names the pinned MD5", asset.md5_hash == NANO_MD5)

W1026 22:41:34.400000 67515 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


weights: rf-detr-nano.pth | licence in the framework's model config: Apache-2.0
CHECK: the Nano weights are published under Apache-2.0: True
CHECK: the framework's registry names the pinned URL: True
CHECK: the framework's registry names the pinned MD5: True


### Checking what we download

The checked fetch of the earlier notebooks: https only, a size limit and a timeout, a temporary file
renamed only after its SHA-256 matched the pin, and a zip validator (no absolute paths, no `..`, no
links, bounded size and ratio) instead of `extractall`. The SHA-256 of the weights was computed once
from the file whose MD5 equals the framework's registry entry; the next cell checks that MD5 again.

In [10]:
COCO128_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip"
# Pinned SHA-256 of every download. A file with any other digest is deleted and the notebook stops.
PINNED_SHA256 = {
    "coco128.zip": "61e5e3028863d8ffc3b81d6a514603954889f0edd5e4b44c4ce60b2da99aeb8e",
    "rf-detr-nano.pth": "d8d6b9ee57d4d0ed2b1f305163624712a0532cb7bce0c747317984fc5457440d",
}
MAX_DOWNLOAD_BYTES = {"coco128.zip": 20_000_000, "rf-detr-nano.pth": 400_000_000}
ZIP_LIMITS = {"files": 1000, "bytes": 50_000_000, "ratio": 20}


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


class HttpsOnlyRedirects(urllib.request.HTTPRedirectHandler):
    """Follow a redirect only to another https URL (a download host may send you to a CDN)."""

    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if not newurl.startswith("https://"):
            raise urllib.error.URLError("a redirect to a non-https URL was refused")
        return super().redirect_request(req, fp, code, msg, headers, newurl)


OPENER = urllib.request.build_opener(HttpsOnlyRedirects)


def fetch(url, name):
    """Download over https into the cache (bounded, via a temporary file) and check the SHA-256.

    A cached file with the wrong digest is deleted and fetched again; a download with the wrong
    digest is deleted and raises. Nothing is unpacked or loaded before this check passes."""
    dest = DATA / name
    if dest.exists() and sha256_of(dest) != PINNED_SHA256[name]:
        dest.unlink()  # never reuse a cached file that does not match
    if not dest.exists():
        if not url.startswith("https://"):
            raise ValueError("only https downloads are allowed")
        # a part file only this process has open (mkstemp: exclusive, mode 0600), never a fixed name
        fd, part_name = tempfile.mkstemp(dir=DATA, prefix=name + ".", suffix=".part")
        part = Path(part_name)
        try:
            with os.fdopen(fd, "wb") as out, OPENER.open(url, timeout=60) as response:
                total = 0
                while chunk := response.read(1 << 20):
                    total += len(chunk)
                    if total > MAX_DOWNLOAD_BYTES[name]:
                        raise RuntimeError(
                            f"{name} is larger than {MAX_DOWNLOAD_BYTES[name]} bytes"
                        )
                    out.write(chunk)
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            part.unlink(missing_ok=True)
            raise RuntimeError(
                f"Could not download {name} ({type(exc).__name__}). This notebook needs the "
                f"internet once. Download {url} yourself, put the file in the folder named by "
                "CVPILOT_NOTEBOOK_DATA (default: ~/.cache/cvpilot-notebooks) and run this cell again."
            ) from None
        except RuntimeError:
            part.unlink(missing_ok=True)
            raise
        digest = sha256_of(part)
        if digest != PINNED_SHA256[name]:
            part.unlink()
            raise RuntimeError(f"{name}: SHA-256 {digest} is not the pinned {PINNED_SHA256[name]}")
        part.replace(dest)  # atomic: a half-downloaded file never has the final name
    print(
        f"{name:12s} {dest.stat().st_size / 1e6:5.1f} MB  sha256 {sha256_of(dest)}  (matches the pin)"
    )
    return dest


def private_copy(source, name, dest=None):
    """Copy a checked download into this run's private folder (WORK, mode 0700) and check the copy's
    SHA-256 again: what is loaded is that copy, which no other user can replace."""
    dest = dest or WORK / "verified" / name
    dest.parent.mkdir(mode=0o700, parents=True, exist_ok=True)
    shutil.copyfile(source, dest)
    digest = sha256_of(dest)
    if digest != PINNED_SHA256[name]:
        dest.unlink()
        raise RuntimeError(f"{name}: the private copy has SHA-256 {digest}, not the pin")
    return dest


def unpacked_matches(zip_path, target):
    """True when ``target`` holds exactly the archive's files, each with the archive's bytes
    (SHA-256), and no link: an unpacked folder is never trusted because it exists."""
    with zipfile.ZipFile(zip_path) as archive:
        expected = {}
        for member in archive.infolist():
            if not member.is_dir():
                with archive.open(member) as handle:
                    expected[member.filename] = hashlib.sha256(handle.read()).hexdigest()
    found = {}
    for path in target.rglob("*"):
        if path.is_symlink():
            return False
        if path.is_file():
            found[path.relative_to(target).as_posix()] = sha256_of(path)
    return found == expected


def safe_unzip(zip_path, target):
    """Unpack into a new folder after checking every member; never `extractall`."""
    if target.exists():
        if unpacked_matches(zip_path, target):
            return
        shutil.rmtree(target)  # anything else than exactly the archive's files: unpack again
    staging = Path(tempfile.mkdtemp(dir=DATA, prefix="unzip-"))
    try:
        with zipfile.ZipFile(zip_path) as archive:
            members = archive.infolist()
            too_big = sum(m.file_size for m in members) > ZIP_LIMITS["bytes"]
            if len(members) > ZIP_LIMITS["files"] or too_big:
                raise RuntimeError("the archive has too many files or is too large")
            for member in members:
                name = Path(member.filename)
                mode = member.external_attr >> 16
                if name.is_absolute() or ".." in name.parts or "\\" in member.filename:
                    raise RuntimeError(f"unsafe name in the archive: {member.filename}")
                if mode and (mode & 0o170000) not in (0o100000, 0o040000):
                    raise RuntimeError(f"not a plain file or folder: {member.filename}")
                ratio = member.file_size / max(member.compress_size, 1)
                if ratio > ZIP_LIMITS["ratio"]:
                    raise RuntimeError(f"suspicious compression ratio: {member.filename}")
            for member in members:
                out = staging / member.filename
                if member.is_dir():
                    out.mkdir(parents=True, exist_ok=True)
                    continue
                out.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as src, open(out, "wb") as dst:
                    shutil.copyfileobj(src, dst)
        staging.rename(target)
    except BaseException:
        shutil.rmtree(staging, ignore_errors=True)
        raise


with timed("downloads"):
    coco_zip = fetch(COCO128_URL, "coco128.zip")
    # a pickle: the framework loads the checked private copy, never the cached file
    nano_path = private_copy(fetch(NANO_URL, "rf-detr-nano.pth"), "rf-detr-nano.pth")
COCO_DIR = DATA / "coco128-unpacked"
safe_unzip(coco_zip, COCO_DIR)
COCO_ROOT = COCO_DIR / "coco128"
IMAGES = sorted((COCO_ROOT / "images" / "train2017").glob("*.jpg"))
print(f"{len(IMAGES)} images in $CVPILOT_NOTEBOOK_DATA/coco128-unpacked/coco128/images/train2017")

coco128.zip    7.0 MB  sha256 61e5e3028863d8ffc3b81d6a514603954889f0edd5e4b44c4ce60b2da99aeb8e  (matches the pin)


rf-detr-nano.pth 366.3 MB  sha256 d8d6b9ee57d4d0ed2b1f305163624712a0532cb7bce0c747317984fc5457440d  (matches the pin)


[downloads: 0.6 s]
128 images in $CVPILOT_NOTEBOOK_DATA/coco128-unpacked/coco128/images/train2017


In [11]:
md5 = hashlib.md5(usedforsecurity=False)
with open(nano_path, "rb") as handle:
    for chunk in iter(lambda: handle.read(1 << 20), b""):
        md5.update(chunk)
print("MD5 of the downloaded weights:", md5.hexdigest())
check("the downloaded weights have the MD5 the framework publishes", md5.hexdigest() == NANO_MD5)

MD5 of the downloaded weights: fb6504cce7fbdc783f7a46991f07639f
CHECK: the downloaded weights have the MD5 the framework publishes: True


### What CV-Pilot's strict reader says about the weights file

The file is a PyTorch zip archive with a pickle inside. CV-Pilot's strict scan reads the pickle's
opcodes without running them and accepts only tensor storage and a short allow-list of names. The
framework loads this file itself, with its own loading code, and this notebook does not check which
loading mode it ends up in. The control used here is the SHA-256 pin, checked again on a private copy
in this run's own folder right before the framework opens that copy.

In [12]:
from cvpilot.ckpt import serializers

raw = nano_path.read_bytes()
try:
    serializers.scan_pt_archive(raw)
    strict = ("accepted", "")
except Exception as exc:  # noqa: BLE001  # the refusal is the result shown here
    strict = ("refused", f"{getattr(exc, 'reason', type(exc).__name__)}: {exc}")
print(f"strict scan: {strict[0]} {strict[1]}")
layout("RF-DETR", "rf-detr-nano.pth (published weights)", *strict)
# Diagnostic only, for this notebook: allow argparse.Namespace for one scan and see whether anything
# else in the file is outside the allow-list. CV-Pilot's own allow-list is not changed.
allowed = serializers._ALLOWED_GLOBALS
serializers._ALLOWED_GLOBALS = allowed | {("argparse", "Namespace")}
try:
    tensor_bytes = serializers.scan_pt_archive(raw)
    widened = f"accepted, {tensor_bytes} bytes of tensor data"
except Exception as exc:  # noqa: BLE001
    tensor_bytes, widened = None, f"refused: {exc}"
finally:
    serializers._ALLOWED_GLOBALS = allowed
del raw
print("with argparse.Namespace allowed for this one scan:", widened)
check(
    "the strict scan refuses the published weights only for argparse.Namespace",
    strict[0] == "refused" and "argparse.Namespace" in strict[1] and tensor_bytes is not None,
)

strict scan: refused unsafe_archive: the pickle refers to argparse.Namespace
with argparse.Namespace allowed for this one scan: accepted, 365606360 bytes of tensor data
CHECK: the strict scan refuses the published weights only for argparse.Namespace: True


**What this shows.** The published weights are outside what CV-Pilot's strict reader accepts, and
the only reason is one `argparse.Namespace` (the training arguments stored with the weights).
CV-Pilot does not load this file itself; the framework does, and only from the private copy whose
SHA-256 matched the pin (the framework's MD5 matched too).

# Part B continued. Fine-tune with the hook attached

COCO128's YOLO labels become the COCO layout RF-DETR reads: 64 photos for training and 32 for
validation (the validation photos also serve as the test split the framework expects). Two epochs on
the CPU at the model's native 384 pixels. The framework counts 80 classes in this data and the
published head has 90 outputs, so it re-initialises the classification head: the accuracy after two
epochs says nothing about RF-DETR, and the run is here to check the hook.

In [13]:
from PIL import Image
from rfdetr.assets.coco_classes import COCO_CLASSES

COCO_IDS = sorted(COCO_CLASSES)  # YOLO class k is the k-th COCO category id
RF_DATA = WORK / "rf-data"
splits = {"train": IMAGES[:64], "valid": IMAGES[-32:], "test": IMAGES[-32:]}
for split, chosen in splits.items():
    folder = RF_DATA / split
    folder.mkdir(parents=True)
    images, annotations = [], []
    for index, photo in enumerate(chosen):
        width, height = Image.open(photo).size
        shutil.copyfile(photo, folder / photo.name)
        images.append({"id": index, "file_name": photo.name, "width": width, "height": height})
        labels = COCO_ROOT / "labels" / "train2017" / f"{photo.stem}.txt"
        for line in labels.read_text().splitlines() if labels.exists() else []:
            cls, cx, cy, bw, bh = line.split()
            bw, bh = float(bw) * width, float(bh) * height
            x, y = float(cx) * width - bw / 2, float(cy) * height - bh / 2
            annotations.append(
                {
                    "id": len(annotations),
                    "image_id": index,
                    "category_id": COCO_IDS[int(cls)],
                    "bbox": [x, y, bw, bh],
                    "area": bw * bh,
                    "iscrowd": 0,
                }
            )
    categories = [{"id": i, "name": COCO_CLASSES[i], "supercategory": "none"} for i in COCO_IDS]
    (folder / "_annotations.coco.json").write_text(
        json.dumps({"images": images, "annotations": annotations, "categories": categories})
    )
    print(f"{split:5s} {len(images):3d} photos, {len(annotations):4d} boxes")

train  64 photos,  542 boxes
valid  32 photos,  225 boxes
test   32 photos,  225 boxes


In [14]:
import rfdetr.training
from rfdetr import RFDETRNano
from rfdetr.training.callbacks import coco_eval

import cvpilot

# The framework draws its metric tables with terminal colour codes; this notebook reads the values
# from the store and from the framework's own metrics.csv instead, so the tables are switched off.
coco_eval._render_summary_tables = lambda *args, **kwargs: None
for name in ("py" + "torch_lightning", "lightning", "lightning_fabric", "rf-detr"):
    logging.getLogger(name).setLevel(logging.ERROR)
FACTORY = rfdetr.training.build_trainer
quiet = io.StringIO()
with contextlib.redirect_stdout(quiet), contextlib.redirect_stderr(quiet):
    model = RFDETRNano(pretrain_weights=str(nano_path))  # the checked private copy
run = cvpilot.autolog(model, project="rfdetr", task="detection")
print("run", run.run_id[:10] + "...", "attached to", type(model).__name__)
check("autolog picked the RF-DETR hook", "train" in vars(model))

run 01M5WAHVMQ... attached to RFDETRNano
CHECK: autolog picked the RF-DETR hook: True


In [15]:
with (
    timed("two epochs of RF-DETR Nano on the CPU"),
    contextlib.redirect_stdout(quiet),
    contextlib.redirect_stderr(quiet),
):
    model.train(
        dataset_dir=str(RF_DATA),
        epochs=2,
        batch_size=4,
        device="cpu",
        num_workers=0,
        output_dir=str(WORK / "rf-out"),
        progress_bar=None,
        seed=7,
    )
check(
    "the framework's build_trainer is its own again after train",
    rfdetr.training.build_trainer is FACTORY,
)

[two epochs of RF-DETR Nano on the CPU: 44.8 s]
CHECK: the framework's build_trainer is its own again after train: True


In [16]:
from cvpilot import runs
from cvpilot.store.local import open_store

view = runs.get_run_view("rfdetr", run.run_id, store_dir=Path("store"))
store = open_store("rfdetr", Path("store"))
points = store.read_points(run.run_id)
store.close()
keys = sorted({p.key for p in points})
print("status:", view.meta.status, "| framework:", view.meta.framework.split()[0])
print(f"{len(points)} points, {len(keys)} keys:", ", ".join(keys))
check("the run is finished when train returns", view.meta.status == "finished")
check(
    "training losses and validation mAP were recorded",
    {"train/loss", "val/mAP_50_95", "val/mAP_50"} <= set(keys),
)

status: finished | framework: rfdetr==1.11.2
46 points, 23 keys: epoch, train/cardinality_error, train/cardinality_error_0, train/cardinality_error_enc, train/class_error, train/loss, train/loss_bbox, train/loss_bbox_aux, train/loss_ce, train/loss_ce_aux, train/loss_giou, train/loss_giou_aux, val/F1, val/ema_mAP_50, val/ema_mAP_50_95, val/ema_mAP_75, val/ema_mAR, val/mAP_50, val/mAP_50_95, val/mAP_75, val/mAR, val/precision, val/recall
CHECK: the run is finished when train returns: True
CHECK: training losses and validation mAP were recorded: True


In [17]:
print(f"{'key':22s} {'step':>5s} {'value':>9s}")
for p in points:
    if p.key in ("train/loss", "val/mAP_50_95", "val/mAP_50", "val/ema_mAP_50_95"):
        print(f"{p.key:22s} {p.step:5d} {p.value:9.4f}")

key                     step     value
val/ema_mAP_50_95         15    0.0165
val/mAP_50_95             15    0.0165
val/mAP_50                15    0.0208
train/loss                15    6.0986
val/ema_mAP_50_95         31    0.0626
val/mAP_50_95             31    0.0626
val/mAP_50                31    0.0734
train/loss                31    5.7938


**What this shows.** The hook reached the trainer that `train` built for itself: the framework's
training loss and its COCO metrics were recorded at the trainer's global step, and the run was
finished when `train` returned. Low mAP is expected here (see above).

### Same numbers as the framework's own log

RF-DETR writes its own `metrics.csv` with Lightning's CSV logger. CV-Pilot's values must be the same
numbers at the same steps.

In [18]:
csv_files = sorted((WORK / "rf-out").rglob("metrics.csv"))
own = {}
for row in csv.DictReader(open(csv_files[0])) if csv_files else []:
    for key in ("val/mAP_50_95", "val/mAP_50", "train/loss"):
        if row.get(key):
            own[(key, int(row["step"]))] = float(row[key])
ours = {
    (p.key, p.step): p.value
    for p in points
    if p.key in ("val/mAP_50_95", "val/mAP_50", "train/loss")
}
shared = sorted(set(own) & set(ours))
worst = max((abs(own[k] - ours[k]) for k in shared), default=float("nan"))
print(
    f"framework log: {len(own)} values, CV-Pilot: {len(ours)} values, {len(shared)} at the same key and step"
)
print("largest difference:", worst)
check(
    "every value both logged at the same key and step is the same number",
    len(shared) >= 2 and worst <= 1e-6,
)

framework log: 6 values, CV-Pilot: 6 values, 6 at the same key and step
largest difference: 0.0
CHECK: every value both logged at the same key and step is the same number: True


### A training that fails, and the files the framework wrote

A second model trains on a folder that does not exist: the framework raises, the exception reaches
the caller unchanged and the run is marked failed. Then each checkpoint file RF-DETR wrote in the
first run goes through CV-Pilot's strict scan.

In [19]:
with contextlib.redirect_stdout(quiet), contextlib.redirect_stderr(quiet):
    broken = RFDETRNano(pretrain_weights=str(nano_path))
failing = cvpilot.autolog(broken, project="rfdetr", task="detection")
try:
    with contextlib.redirect_stdout(quiet), contextlib.redirect_stderr(quiet):
        broken.train(
            dataset_dir=str(WORK / "no-such-folder"),
            epochs=1,
            batch_size=4,
            device="cpu",
            num_workers=0,
            output_dir=str(WORK / "rf-out-2"),
            progress_bar=None,
        )
    raised = None
except Exception as exc:  # noqa: BLE001  # the exception is the result shown here
    raised = type(exc).__name__
status = runs.get_run_view("rfdetr", failing.run_id, store_dir=Path("store")).meta.status
print("train raised:", raised, "| run status:", status)
check(
    "a failing train() raises unchanged and marks the run failed",
    raised is not None and status == "failed",
)

train raised: AssertionError | run status: failed
CHECK: a failing train() raises unchanged and marks the run failed: True


In [20]:
for path in sorted((WORK / "rf-out").glob("*.p*")) + sorted((WORK / "rf-out").glob("*.ckpt")):
    data = path.read_bytes()
    try:
        serializers.scan_pt_archive(data)
        outcome = ("accepted", "")
    except Exception as exc:  # noqa: BLE001
        outcome = ("refused", f"{getattr(exc, 'reason', type(exc).__name__)}: {exc}")
    print(f"{path.name:32s} {len(data) / 1e6:7.1f} MB  {outcome[0]:8s} {outcome[1][:90]}")
    layout("RF-DETR", path.name, *outcome)

checkpoint_best_ema.pth            122.0 MB  accepted 
checkpoint_best_total.pth          122.0 MB  accepted 
last_ema.pth                       122.0 MB  accepted 


last.ckpt                          487.7 MB  accepted 


In [21]:
sh("cvpilot runs list --project rfdetr")

$ cvpilot runs list --project rfdetr
RUN_ID                      STATUS     STARTED                    NAME
01M5WAK9S1DYJNCNT0XN6YV8ZT  failed     2026-10-26T14:42:27.026330+00:00  rfdetr-2026-10-26
01M5WAHVMQZDK30Y128ZF2AMEX  finished   2026-10-26T14:41:39.766266+00:00  rfdetr-2026-10-26
exit code: 0


**What this shows.** The run list with the finished and the failed run, and what the strict reader
says about the files the framework wrote during training. The hook still offers no resume helper:
the framework's `train(resume=...)` reads the file with its own loader in a mode that falls back to
full unpickling when the restricted load fails, so a file swapped after CV-Pilot verified it could
be unpickled. Use `CheckpointManager` for state you want verified.

# Summary

In [22]:
print(f"{'framework':10s} {'file':40s} {'outcome':9s} message")
for framework, file, outcome, message in LAYOUTS:
    print(f"{framework:10s} {file[:40]:40s} {outcome:9s} {message[:110]}")
print()
failed = [label for label, ok in CHECKS if not ok]
print(f"checks: {len(CHECKS)}, failed: {len(failed)}", failed)
print("timings (s):", TIMINGS)

framework  file                                     outcome   message
RF-DETR    rf-detr-nano.pth (published weights)     refused   unsafe_archive: the pickle refers to argparse.Namespace
RF-DETR    checkpoint_best_ema.pth                  accepted  
RF-DETR    checkpoint_best_total.pth                accepted  
RF-DETR    last_ema.pth                             accepted  
RF-DETR    last.ckpt                                accepted  

checks: 20, failed: 0 []
timings (s): {'1,000 kills': 198.0, '200 cut uploads': 7.8, 'downloads': 0.6, 'two epochs of RF-DETR Nano on the CPU': 44.8}


### Clean up

In [23]:
# Remove this run's scratch folder (stores, checkpoints, copies of the weights); the download
# cache is kept so the next run needs no download.
os.chdir(Path.home())  # leave the folder that is about to be removed
shutil.rmtree(WORK, ignore_errors=True)
print("this run's scratch folder removed:", not WORK.exists())

this run's scratch folder removed: True
